In [2]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Spark Job Progress Monitor already enabled


In [3]:
#Reading-> Commo-Control
Como_Result_Final1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_comorbidity_Paired")

In [4]:
Como_Result_Final1.createOrReplaceTempView('Epilepsy_Control_Como')

In [5]:
#Reading-> Commo-Cohort
Como_Result_Final2 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_comorbidity_Paired")

In [6]:
Como_Result_Final2.createOrReplaceTempView('Epilepsy_Cohort_Como')

In [7]:
##############################################Final dictionary to be used for control commo conversion process #########################
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# Read the CSV file into a DataFrame
df = spark.read.csv("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/ICDconvertion.csv", header=True, inferSchema=True)

# Define a UDF to cast the "icd9cm" column to StringType
def cast_to_string(icd9cm):
    return str(icd9cm)

# Register the UDF
cast_to_string_udf = F.udf(cast_to_string, StringType())

# Apply the UDF to cast "icd9cm" to StringType
df = df.withColumn("icd9cm", cast_to_string_udf(df["icd9cm"]))

# Convert DataFrame to a Pandas DataFrame
pandas_df = df.toPandas()

# Convert Pandas DataFrame to a dictionary
icd_dict = pandas_df.set_index("icd9cm")["icd10cm"].to_dict()

# Show the resulting dictionary
print(icd_dict)

{'10': 'A000', '11': 'A001', '19': 'A009', '20': 'A0100', '21': 'A011', '22': 'A012', '23': 'A013', '29': 'A014', '30': 'A020', '31': 'A021', '320': 'A360', '321': 'A361', '322': 'A3689', '323': 'A362', '324': 'A0224', '329': 'A369', '38': 'A028', '39': 'A029', '40': 'A030', '41': 'A031', '42': 'B20', '43': 'A033', '48': 'A880', '49': 'A039', '50': 'A050', '51': 'A051', '52': 'A052', '53': 'A058', '54': 'A053', '581': 'A055', '589': 'A058', '59': 'A059', '60': 'A060', '61': 'A90', '62': 'A062', '63': 'A064', '64': 'A852', '65': 'A066', '66': 'A067', '68': 'A0689', '69': 'A069', '70': 'A070', '71': 'A829', '72': 'A073', '73': 'A078', '74': 'A072', '75': 'B2790', '78': 'A078', '79': 'A079', '800': 'A044', '801': 'A040', '802': 'A041', '803': 'A042', '804': 'A043', '809': 'A044', '81': 'A048', '82': 'A048', '83': 'A048', '841': 'B519', '842': 'B529', '843': 'B530', '844': 'B538', '845': 'B529', '846': 'B54', '847': 'B538', '849': 'B528', '85': 'A049', '861': 'B575', '862': 'B571', '863': 

In [8]:
Como_Result_Test = spark.sql(""" 
    SELECT
        DISTINCT
        comorbidityid   
    FROM
        Epilepsy_Control_Como
""")
Como_Result_Test.show()
# # # Write the distinct comorbidityid values to a CSV file
# # Como_Result_Test.write.csv("comorbidityid_distinct.csv", header=True)
# # Como_Result_Test.write.csv("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/comorbidityid_distinct.csv", header=True)

# Collect the distinct comorbidityid values into a list
comorbidityid_list = Como_Result_Test.rdd.flatMap(lambda x: x).collect()

# Show the distinct comorbidityid values
print(comorbidityid_list)

+-------------+
|comorbidityid|
+-------------+
|      Z87.820|
|        850.9|
|       E83.42|
|       Z76.89|
|        959.4|
|      F12.929|
|       810.03|
|     S83.231A|
|       787.20|
|    133933007|
|       H52.02|
|       H55.81|
|       M62.89|
|        286.9|
|          Z21|
|        840.8|
|       K85.90|
|       M51.87|
|       M23.92|
|        G03.9|
+-------------+
only showing top 20 rows

['Z87.820', '850.9', 'E83.42', 'Z76.89', '959.4', 'F12.929', '810.03', 'S83.231A', '787.20', '133933007', 'H52.02', 'H55.81', 'M62.89', '286.9', 'Z21', '840.8', 'K85.90', 'M51.87', 'M23.92', 'G03.9', 'M06.09', '803.00', 'N99.89', 'E937.8', 'M24.011', 'M77.40', 'S62.350A', '415.11', 'E006.9', 'M43.10', 'V14.2', 'N81.4', 'B17.9', '732.1', '521.81', '807.05', '783.43', 'T46.0X4A', 'Z51.11', 'Z45.89', 'N35.911', '110.9', 'S60.021A', '536.8', 'Q27.30', '958.3', 'V42.3', 'S60.012S', 'Z59.6', '733.01', 'F15.11', 'S14.125A', 'M50.31', 'V17.1', 'N60.31', 'S63.681A', 'M67.441', 'J30.9, R09.82'

In [9]:
Como_Result_Test = spark.sql(""" 
    SELECT
        DISTINCT
        comorbidityid   
    FROM
        Epilepsy_Cohort_Como
""")
Como_Result_Test.show()
# # # Write the distinct comorbidityid values to a CSV file
# # Como_Result_Test.write.csv("comorbidityid_distinct.csv", header=True)
# # Como_Result_Test.write.csv("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/comorbidityid_distinct.csv", header=True)

# Collect the distinct comorbidityid values into a list
comorbidityid_list1 = Como_Result_Test.rdd.flatMap(lambda x: x).collect()

# Show the distinct comorbidityid values
print(comorbidityid_list1)

+-------------+
|comorbidityid|
+-------------+
|       M23.92|
|      Z87.820|
|     S83.231A|
|       E83.42|
|        850.9|
|       787.20|
|       Z76.89|
|        R51.0|
|        V14.2|
|       H55.81|
|       M62.89|
|       Z51.11|
|       M06.09|
|        V17.1|
|        Z59.6|
|       K85.90|
|          Z21|
|       M50.31|
|        N81.4|
|       997.62|
+-------------+
only showing top 20 rows

['M23.92', 'Z87.820', 'S83.231A', 'E83.42', '850.9', '787.20', 'Z76.89', 'R51.0', 'V14.2', 'H55.81', 'M62.89', 'Z51.11', 'M06.09', 'V17.1', 'Z59.6', 'K85.90', 'Z21', 'M50.31', 'N81.4', '997.62', 'S60.021A', '959.4', 'M67.441', '521.81', 'M43.10', 'K66.8', '286.9', 'N35.911', 'S59.919A', 'M86.8X0', 'S02.610A', 'F12.929', '803.00', '840.8', 'M31.6', 'N99.89', 'H35.32', 'S63.592A', 'C84.60', '783.43', '536.8', 'O36.8990', 'B17.9', '66857006', 'Q27.30', 'E006.9', 'Q79.9', 'M77.40', 'S52.209D', '191.9', 'L89.302', 'Y90.4', 'V57.6XXA', '458.29', 'Z45.89', 'F15.11', 'G03.9', 'S63.681A', 'J2

In [10]:
# Combine the lists and extract distinct values
combined_and_distinct = list(set(comorbidityid_list + comorbidityid_list1))
# print(combined_and_distinct)
# # Print the distinct values
# for item in combined_and_distinct:
#     print(item)
# Iterate through the list and print values containing commas
for item in combined_and_distinct:
    if ',' in item:
        print(item)

S61.459A, W54.0XXA
J18.9, J98.4
S56.222S, S51.002S
M54.9, M79.605
Z98.890, Z90.49
I27.29, I26.99
O34.80, N81.10
Z12.2, F17.200
E11.621, L97.509
S09.90XD, S06.9X9D
M43.00, M43.10
E66.9, Z68.35
S61.052A, W54.0XXA
E11.621, L97.509, Z79.4
M25.541, M25.542
E11.641, Z79.4
G95.9, M54.12
M79.601, G89.29
R10.12, G89.29
O99.612, K59.00
M79.601, M79.602
T82.6XXD, I38
S81.852A, W55.01XA
I63.9, R29.810
S91.051A, W54.0XXA
M54.42, G89.29
S09.90XD, V19.9XXD
H01.002, H01.005
S01.81XA, S01.112A
E11.22, N18.4
M79.604, G89.29
S52.521A, S52.621A
S06.9X9S, F01.51
N60.01, N60.02
H91.93, S09.90XS
M81.0, Z79.899
E08.8, Z79.4
S02.81XD, S02.82XD
T56.0X1D, M1A.10X0
E88.09, E46
E66.3, Z68.28
N40.1, R35.1
T81.4XXD, A41.9
G20, F02.80
R10.31, G89.29
M24.541, M24.542
S02.91XA, S06.9X9A
F80.9, F09
H01.021, H01.024
S32.433A, S32.443A
M25.571, G89.29
G31.89, F09, S06.9X9S
T81.718A, I26.99
M21.6X1, M21.6X2
T82.7XXA, R78.81
E11.319, E11.65
E66.09, Z68.54
M25.522, M25.422
M79.A21, M79.A22
S05.02XA, H44.002
M79.661, M79.662


In [ ]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType

# Define a Python function to process comorbidityid
def process_comorbidityid(comorbidityid):
    # Check if comorbidityid contains a decimal point
    if '.' in comorbidityid:
        # Remove the decimal point
        comorbidityid = comorbidityid.replace('.', '')
    
    # Look up comorbidityid in the dictionary and return the mapped value if found
    mapped_value = icd_dict.get(comorbidityid, None)
    if mapped_value is not None:
        return mapped_value
    
    # If no match found, retain the original comorbidityid
    return comorbidityid

# Register the process_comorbidityid function as a UDF
process_comorbidityid_udf = udf(process_comorbidityid, StringType())

# Apply the UDF to replace comorbidityid values
result_df_full = Como_Result_Final1.withColumn("comorbidityid", process_comorbidityid_udf(col("comorbidityid")))

# Show the resulting DataFrame
result_df_full.show(truncate=False)

In [11]:
########################Final Package to be used #####################################################################3
from pyspark.sql.functions import col, udf, split, regexp_replace, explode
from pyspark.sql.types import StringType
from pyspark.sql import SparkSession

# Define a Python function to process comorbidityid
def process_comorbidityid(comorbidityid):
    # Check if comorbidityid contains a comma
    if ',' in comorbidityid:
        # Split comorbidityid by commas
        comorbidity_list = comorbidityid.split(',')
        # Process individual values and map them to the dictionary
        processed_comorbidities = [process_individual_comorbidity(cid) for cid in comorbidity_list]
        # Return the processed values as a comma-separated string
        return ','.join(processed_comorbidities)
    
    # Check if comorbidityid contains a decimal point
    if '.' in comorbidityid:
        # Remove the decimal point
        comorbidityid = comorbidityid.replace('.', '')

    # Look up comorbidityid in the dictionary and return the mapped value if found
    mapped_value = icd_dict.get(comorbidityid, None)
    if mapped_value is not None:
        return mapped_value  # Remove square brackets and return as is

    # If no match found, retain the original comorbidityid
    return comorbidityid

# Define a function to process individual comorbidity values
def process_individual_comorbidity(comorbidity):
    # Remove decimal points
    comorbidity = comorbidity.replace('.', '')
    # Strip square brackets and leading/trailing spaces
    comorbidity = comorbidity.strip('[]').strip()
    # Look up comorbidityid in the dictionary and return the mapped value if found
    mapped_value = icd_dict.get(comorbidity, None)
    if mapped_value is not None:
        return mapped_value
    # If no match found, retain the original comorbidity
    return comorbidity

# Register the process_comorbidityid function as a UDF
process_comorbidityid_udf = udf(process_comorbidityid, StringType())

# Assuming you have a DataFrame named Como_Result_Final1
# Split the comorbidityid values by comma and apply the UDF to each element
result_df_full = Como_Result_Final1.withColumn("comorbidityid", 
    split(col("comorbidityid"), ",").cast(StringType())
).withColumn("comorbidityid", process_comorbidityid_udf(col("comorbidityid")))

# Remove square brackets from comorbidityid values
result_df_full = result_df_full.withColumn("comorbidityid", regexp_replace(col("comorbidityid"), r'\[|\]', ''))
# Split the comorbidityid values by comma and explode the resulting array
result_df_full = result_df_full.withColumn("comorbidityid", split(col("comorbidityid"), ",")) \
              .withColumn("comorbidityid", explode(col("comorbidityid")))
# Show the resulting DataFrame
result_df_full.show(truncate=False)

+------------------------------------+-------------+
|personid                            |comorbidityid|
+------------------------------------+-------------+
|0009ad92-088d-4991-82b6-17569e395480|S0993XA      |
|002f936d-b037-455e-b331-9980269cc611|H6690        |
|002f936d-b037-455e-b331-9980269cc611|R509         |
|002f936d-b037-455e-b331-9980269cc611|J40          |
|002f936d-b037-455e-b331-9980269cc611|Z20822       |
|002f936d-b037-455e-b331-9980269cc611|K2970        |
|002f936d-b037-455e-b331-9980269cc611|J020         |
|002f936d-b037-455e-b331-9980269cc611|S060X0A      |
|002f936d-b037-455e-b331-9980269cc611|X58XXXD      |
|002f936d-b037-455e-b331-9980269cc611|Z4802        |
|002f936d-b037-455e-b331-9980269cc611|S61203D      |
|002f936d-b037-455e-b331-9980269cc611|W260XXA      |
|002f936d-b037-455e-b331-9980269cc611|S61213A      |
|002f936d-b037-455e-b331-9980269cc611|S61215A      |
|0033e92f-00ac-4bde-a16e-8cf525819ab4|Z1211        |
|0033e92f-00ac-4bde-a16e-8cf525819ab4|9248    

In [12]:
###############################Final Code to be used for conversion of codes with excel provided for commo control ####################
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType

# Define a Python function to convert comorbidityid
def convert_comorbidityid(comorbidityid):
    # Check if comorbidityid contains a decimal point
    if '.' in comorbidityid:
        parts = comorbidityid.split('.')
        key = parts[0]
        mapped_value = icd_dict.get(key, None)
        if mapped_value:
            mapped_value += '.' + '.'.join(parts[1:])
            return mapped_value
    else:
        # Preserve the original value if not found in the dictionary
        mapped_value = icd_dict.get(comorbidityid, None)
        if mapped_value is not None:
            return mapped_value

    # If no mapped value is found or no decimal point, preserve the original value
    return comorbidityid

# Register the convert_comorbidityid function as a UDF
convert_comorbidityid_udf = udf(convert_comorbidityid, StringType())

# Apply the UDF to replace comorbidityid values
result_df = Como_Result_Final1.withColumn("comorbidityid", convert_comorbidityid_udf(col("comorbidityid")))

# Show the resulting DataFrame
result_df.show(truncate=False)

+------------------------------------+-------------+
|personid                            |comorbidityid|
+------------------------------------+-------------+
|0009ad92-088d-4991-82b6-17569e395480|S09.93XA     |
|002f936d-b037-455e-b331-9980269cc611|H66.90       |
|002f936d-b037-455e-b331-9980269cc611|R50.9        |
|002f936d-b037-455e-b331-9980269cc611|J40          |
|002f936d-b037-455e-b331-9980269cc611|Z20.822      |
|002f936d-b037-455e-b331-9980269cc611|K29.70       |
|002f936d-b037-455e-b331-9980269cc611|J02.0        |
|002f936d-b037-455e-b331-9980269cc611|S06.0X0A     |
|002f936d-b037-455e-b331-9980269cc611|X58.XXXD     |
|002f936d-b037-455e-b331-9980269cc611|Z48.02       |
|002f936d-b037-455e-b331-9980269cc611|S61.203D     |
|002f936d-b037-455e-b331-9980269cc611|W26.0XXA     |
|002f936d-b037-455e-b331-9980269cc611|S61.213A     |
|002f936d-b037-455e-b331-9980269cc611|S61.215A     |
|0033e92f-00ac-4bde-a16e-8cf525819ab4|Z12.11       |
|0033e92f-00ac-4bde-a16e-8cf525819ab4|924.8   

In [41]:
result_df_full.createOrReplaceTempView('Epilepsy_Control_RP')

▸,:,


In [ ]:
##################################Final Control Commo Replaced Codes ############################################################
result_df.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_control_commo")

In [13]:
Control_Commo_Updated = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_control_commo")

In [36]:
Como_Result_Final = spark.sql(""" SELECT * from Epilepsy_Control_Como where comorbidityid like '%, %'""")
Como_Result_Final.show(truncate = False)

▸,:,


<IPython.core.display.Javascript object>

+------------------------------------+--------------------------------------+
|personid                            |comorbidityid                         |
+------------------------------------+--------------------------------------+
|08b10128-38d0-4d6b-aea3-3e0b0902b54e|M54.42, M54.41, G89.29                |
|40db8ccb-90a4-4f62-bff1-11c2f33b2cf1|M54.6, G89.29                         |
|4bab70bb-0870-4ada-97fc-cfda4bb61f4a|M79.644, M79.645                      |
|6d5c0b04-2732-4a02-b524-8fdb2b6ba412|M25.561, M25.562, G89.29              |
|6f59a82c-c818-4b77-87fa-83b962a79b09|J03.80, B27.90                        |
|7cf0e170-cab0-4725-9dcf-4977b5fc9a41|M21.6X1, M21.6X2                      |
|7cf0e170-cab0-4725-9dcf-4977b5fc9a41|M21.41, M21.42                        |
|7e56880f-1fc8-435f-887c-c348b7413e0d|H91.91, S02.19XS                      |
|aef0626d-7653-4619-b077-ecb70d8b4a91|S02.40ED, S02.81XD, S02.40CD, S02.31XD|
|df45fd7c-1423-4f37-aac5-2c387270e574|H66.91, H72.91            

<IPython.core.display.Javascript object>

In [42]:
Como_Result_Final = spark.sql(""" SELECT * from Epilepsy_Control_RP where personid = 'aef0626d-7653-4619-b077-ecb70d8b4a91' """)
Como_Result_Final.show(truncate = False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------+
|personid                            |comorbidityid|
+------------------------------------+-------------+
|aef0626d-7653-4619-b077-ecb70d8b4a91|S129XXD      |
|aef0626d-7653-4619-b077-ecb70d8b4a91|3369         |
|aef0626d-7653-4619-b077-ecb70d8b4a91|G959         |
|aef0626d-7653-4619-b077-ecb70d8b4a91|S0240ED      |
|aef0626d-7653-4619-b077-ecb70d8b4a91|S0281XD      |
|aef0626d-7653-4619-b077-ecb70d8b4a91|S0240CD      |
|aef0626d-7653-4619-b077-ecb70d8b4a91|S0231XD      |
|aef0626d-7653-4619-b077-ecb70d8b4a91|S42115A      |
|aef0626d-7653-4619-b077-ecb70d8b4a91|S42102D      |
|aef0626d-7653-4619-b077-ecb70d8b4a91|S42102A      |
|aef0626d-7653-4619-b077-ecb70d8b4a91|S12690A      |
|aef0626d-7653-4619-b077-ecb70d8b4a91|9599         |
|aef0626d-7653-4619-b077-ecb70d8b4a91|S0292XA      |
|aef0626d-7653-4619-b077-ecb70d8b4a91|S12590A      |
|aef0626d-7653-4619-b077-ecb70d8b4a91|F32A         |
|aef0626d-7653-4619-b077-ecb70d8b4a91|S060X9A 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
Como_Result_Final1 = spark.sql(""" SELECT count(personid) from Epilepsy_Control_Como """)
Como_Result_Final1.show(truncate = False)

In [18]:
#Reading-> Commo-Control
result_df_1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_control_commo")

In [19]:
result_df_1.createOrReplaceTempView('Epilepsy_Control_RP1')

In [20]:
#Step3
from pyspark.sql.functions import when, regexp_extract, col, trim

Result_df_rd = spark.sql("""
    SELECT 
        personid,
        CASE 
            WHEN TRIM(comorbidityid) RLIKE '\\.' THEN regexp_extract(TRIM(comorbidityid), '^[^.]+', 0)
            ELSE comorbidityid
        END AS comorbidityid
    FROM Epilepsy_Control_RP
""")
Result_df_rd.show(3, truncate=False)

AnalysisException: 'Table or view not found: Epilepsy_Control_RP; line 8 pos 9'

In [44]:
##################################Final Control Commo Replaced Codes ############################################################-Removed Commas and splitted
Result_df_rd.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_control_commo_fullvalRMCM")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
##################################Final Control Commo Replaced Codes ############################################################
Result_df_rd.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_control_commo_nm")

In [21]:
#Reading-> Commo-Cohort-nm
# normalized_cohort = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_cohort_commo_nm")
# normalized_cohort = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_cohort_commo_fullval")
#Reading-> Commo-Cohort-nm - split commas
normalized_cohort = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_cohort_commo_RMCM")

In [22]:
#Reading-> Commo-Control-nm
# normalized_cohort = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_cohort_commo_nm")
# normalized_control = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_control_commo_fullval")
#Reading-> Commo-Control-nm - split commas
normalized_control = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_control_commo_fullvalRMCM")

In [ ]:
normalized_control.createOrReplaceTempView('Epilepsy_Control_Como_nm')
Como_Result_Final61 = spark.sql(""" SELECT distinct * from Epilepsy_Control_Como_nm where personid = '016bc6bd-3363-4555-ae1e-38e558084f8f' and comorbidityid = 'S060X0A' """)
Como_Result_Final61.show(truncate = False)
# normalized_control.createOrReplaceTempView('Epilepsy_Cohort_Como_nm')
# Como_Result_Final61 = spark.sql(""" SELECT * from Epilepsy_Cohort_Como_nm where personid = '4ed37111-48cc-490e-8ca9-235a554620f8' """)
# Como_Result_Final61.show(truncate = False)

In [ ]:
Como_Result_Final62 = spark.sql(""" SELECT * from Epilepsy_Control_Como where comorbidityid like '850.%' """)
Como_Result_Final62.show(truncate = False)

In [ ]:
Como_Result_Final63 = spark.sql(""" SELECT * from Epilepsy_Control_Como where comorbidityid like 'S63.592A' """)
Como_Result_Final63.show(truncate = False)

In [ ]:
Como_Result_Final64 = spark.sql(""" SELECT * from Epilepsy_Control_Como_nm where comorbidityid like 'S63592A' """)
Como_Result_Final64.show(truncate = False)

In [ ]:
#Reading-> Commo-Control-nm
# normalized_control = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_control_commo_nm")
normalized_control = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_control_commo_fullval")

In [23]:
from pyspark.sql.functions import col, count, countDistinct

# Filter the DataFrame to select only rows where "comorbidityid" starts with an alphabet
filtered_df_cohort = normalized_cohort.filter(col("comorbidityid").rlike('^[A-Za-z]'))
# Filter the DataFrame to select only rows where "comorbidityid" starts with an alphabet
filtered_df_control = normalized_control.filter(col("comorbidityid").rlike('^[A-Za-z]'))

# Count both unique and non-unique values of "comorbidityid"
count_df = filtered_df_cohort.groupBy("comorbidityid").agg(count("*").alias("count"))

# Show the unique count and non-unique count
unique_count = count_df.count()
total_count = filtered_df_cohort.count()

print(f"Unique Count of cohort: {unique_count}")
print(f"Total Count of cohort(including duplicates): {total_count}")

# Count both unique and non-unique values of "comorbidityid"
count_df1 = filtered_df_control.groupBy("comorbidityid").agg(count("*").alias("count"))

# Show the unique count and non-unique count
unique_count1 = count_df1.count()
total_count1 = filtered_df_control.count()

print(f"Unique Count of control: {unique_count1}")
print(f"Total Count of control(including duplicates): {total_count1}")

# Combine the comorbidityid values from both DataFrames
combined_df = filtered_df_cohort.union(filtered_df_control)

# Count the unique comorbidityid values
unique_count = combined_df.select("comorbidityid").distinct().count()

# Show the unique count
print(f"Unique Comorbidity Count (Combined): {unique_count}")

Unique Count of cohort: 38998
Total Count of cohort(including duplicates): 6900477
Unique Count of control: 41448
Total Count of control(including duplicates): 17771072
Unique Comorbidity Count (Combined): 47388


In [25]:
# Select distinct comorbidityid values and collect them into a list
unique_comorbidity_list = combined_df.select("comorbidityid").distinct().rdd.map(lambda x: x[0]).collect()
print(unique_comorbidity_list)
# # Show the list of unique comorbidityid values
# print("Unique Comorbidity ID List:")
# for comorbidityid in unique_comorbidity_list:
#     print(comorbidityid)
# import pandas as pd

# # Select distinct comorbidityid values and collect them into a list
# unique_comorbidity_list = combined_df.select("comorbidityid").distinct().rdd.map(lambda x: x[0]).collect()

# # Create a DataFrame from the list
# df = pd.DataFrame({"Unique Comorbidity ID": unique_comorbidity_list})

# # Define the Excel file path
# excel_file = "unique_comorbidity_ids.xlsx"

# # Export the DataFrame to an Excel file
# df.to_excel(excel_file, index=False, encoding='latin1')

# print(f"Unique comorbidity IDs have been exported to {excel_file}")

['N179', 'R5382', 'R791', 'Z950', 'S27321A', 'Z3492', 'S61219A', 'Z87440', 'O26833', 'R159', 'O98513', 'S32402A', 'I676', 'R41841', 'V491', 'G930', 'Z21', 'N921', 'M5031', 'F900', 'S82131A', 'N942', 'S98929A', 'Y93E2', 'H5213', 'S7000XA', 'Q357', 'M67432', 'S92919D', 'V1551', 'C609', 'Z972', 'V890XXA', 'S86111D', 'V4575', 'Z0379', 'E8182', 'G527', 'M23306', 'T85193A', 'Y248XXA', 'T24309A', 'I82413', 'S12190K', 'Y92001', 'T8541XA', 'T465X1A', 'J689', 'P761', 'P551', 'T43625A', 'S0120XA', 'N140', 'M2610', 'Z2913', 'E8159', 'G971', 'O281', 'F459', 'S2001XA', 'Q8719', 'T8242XA', 'V239XXD', 'L010', 'M8730', 'S60551A', 'V284', 'S02400B', 'V00818A', 'S99922S', 'S92302G', 'H4031X0', 'M5081', 'F985', 'V988XXS', 'T25211S', 'S63610A', 'M00861', 'T382X5A', 'S82209D', 'S63219D', 'Z9629', 'O9A511', 'J1083', 'Z90712', 'M65251', 'C718', 'S40871A', 'S5332XD', 'T82828A', 'N2886', 'S8251XB', 'L743', 'S32312A', 'Y730', 'S63694A', 'M23302', 'T8460XA', 'S82829A', 'D4862', 'S12690D', 'E749', 'G5771', 'S903',

In [27]:
# Filter the DataFrame to select records where "comorbidityid" ends with "XRA"
filtered_df = combined_df.filter(col("comorbidityid").rlike(r'XRA$'))

# Count the number of matching records
count_with_XRA = filtered_df.count()
filtered_df.show()
# Show the count
print(f"Number of Records with 'XRA' in Comorbidity ID: {count_with_XRA}")

+--------------------+--------------------+
|            personid|       comorbidityid|
+--------------------+--------------------+
|0b901cd2-130d-4f4...|ABjfrwDwaSD2ZUcT2...|
|7dfebd9b-31cd-4ac...|ABjfrwDwaSDkPxz02...|
|f2dc625b-50a6-474...|ABjfrwDwaSD2ZV2d2...|
|2fdc8b4c-e92f-44d...|ABjfrwDwaSDSLHqv2...|
|b862ca95-0cc6-4ab...|ABjfrwDwaSD2ZV2d2...|
|ce38e7ae-5cfe-47a...|ABjfrwDwaSD2ZV2d2...|
|7c1d655d-fc6d-448...|ABjfrwDwaSD2ZV2d2...|
|50b363d1-67e1-493...|ABjfrwDwaSD2ZX8X2...|
|32e5bb2e-f350-4e5...|ABjfrwDwaSDkP+Lk2...|
|458ff774-3c02-4bc...|ABjfrwDwaSDSLHqv2...|
|1d7d7aec-26e6-477...|ABjfrwDwaSDkPxSo2...|
|d0e04da8-0622-47c...|ABjfrwDwaSD2Zadr2...|
|a5ff46a3-f4f5-40c...|ABjfrwDwaSDkP5bu2...|
|97e51ef4-beda-4d6...|ABjfrwDwaSD2ZV2d2...|
|4f9aaff2-ceed-4a0...|ABjfrwDwaSD2ZV2d2...|
|61061acf-6e27-4bc...|ABjfrwDwaSD2ZV2d2...|
|dea57484-0d4f-467...|ABjfrwDwaSDSLHqv2...|
|44d13a5b-8b61-454...|ABjfrwDwaSD2ZREz2...|
|c44a357d-0808-4ac...|ABjfrwDwaSD2ZV2d2...|
|8625f8d4-8ae9-495...|ABjfrwDwaS

In [28]:
# Assuming you have a DataFrame named combined_df with a "comorbidityid" column

# Filter the DataFrame to exclude "comorbidityid" values that end with "XRA"
filtered_df = combined_df.filter(~col("comorbidityid").rlike(r'XRA$'))

# Select distinct comorbidityid values from the filtered DataFrame
unique_comorbidity_list = filtered_df.select("comorbidityid").distinct().rdd.map(lambda x: x[0]).collect()

# Show the list of unique comorbidityid values (excluding those ending with "XRA")
# print("Unique Comorbidity ID List (Excluding 'XRA' Endings):")
print(unique_comorbidity_list)
# for comorbidityid in unique_comorbidity_list:
#     print(comorbidityid)

['N179', 'R5382', 'R791', 'Z950', 'S27321A', 'Z3492', 'S61219A', 'Z87440', 'O26833', 'R159', 'O98513', 'S32402A', 'I676', 'R41841', 'V491', 'G930', 'Z21', 'N921', 'M5031', 'F900', 'S82131A', 'N942', 'S98929A', 'Y93E2', 'H5213', 'S7000XA', 'Q357', 'M67432', 'S92919D', 'V1551', 'C609', 'Z972', 'V890XXA', 'S86111D', 'V4575', 'Z0379', 'E8182', 'G527', 'M23306', 'T85193A', 'Y248XXA', 'T24309A', 'I82413', 'S12190K', 'Y92001', 'T8541XA', 'T465X1A', 'J689', 'P761', 'P551', 'T43625A', 'S0120XA', 'N140', 'M2610', 'Z2913', 'E8159', 'G971', 'O281', 'F459', 'S2001XA', 'Q8719', 'T8242XA', 'V239XXD', 'L010', 'M8730', 'S60551A', 'V284', 'S02400B', 'V00818A', 'S99922S', 'S92302G', 'H4031X0', 'M5081', 'F985', 'V988XXS', 'T25211S', 'S63610A', 'M00861', 'T382X5A', 'S82209D', 'S63219D', 'Z9629', 'O9A511', 'J1083', 'Z90712', 'M65251', 'C718', 'S40871A', 'S5332XD', 'T82828A', 'N2886', 'S8251XB', 'L743', 'S32312A', 'Y730', 'S63694A', 'M23302', 'T8460XA', 'S82829A', 'D4862', 'S12690D', 'E749', 'G5771', 'S903',

In [29]:
# Assuming you have a DataFrame named combined_df with a "comorbidityid" column

# Filter the DataFrame to exclude "comorbidityid" values that end with "XRA"
filtered_df = combined_df.filter(~col("comorbidityid").rlike(r'XRA$'))

# Select distinct comorbidityid values from the filtered DataFrame
unique_comorbidity_list = filtered_df.select("comorbidityid").distinct().rdd.map(lambda x: x[0]).collect()

# Create a new list with the modified comorbidityid values
modified_comorbidity_list = [item[:3] + "." + item[3:] for item in unique_comorbidity_list]
print(modified_comorbidity_list)

# # Show the list of unique comorbidityid values with floating points (excluding those ending with "XRA")
# print("Unique Comorbidity ID List with Floating Points (Excluding 'XRA' Endings):")
# for comorbidityid in modified_comorbidity_list:
#     print(comorbidityid)

['N17.9', 'R53.82', 'R79.1', 'Z95.0', 'S27.321A', 'Z34.92', 'S61.219A', 'Z87.440', 'O26.833', 'R15.9', 'O98.513', 'S32.402A', 'I67.6', 'R41.841', 'V49.1', 'G93.0', 'Z21.', 'N92.1', 'M50.31', 'F90.0', 'S82.131A', 'N94.2', 'S98.929A', 'Y93.E2', 'H52.13', 'S70.00XA', 'Q35.7', 'M67.432', 'S92.919D', 'V15.51', 'C60.9', 'Z97.2', 'V89.0XXA', 'S86.111D', 'V45.75', 'Z03.79', 'E81.82', 'G52.7', 'M23.306', 'T85.193A', 'Y24.8XXA', 'T24.309A', 'I82.413', 'S12.190K', 'Y92.001', 'T85.41XA', 'T46.5X1A', 'J68.9', 'P76.1', 'P55.1', 'T43.625A', 'S01.20XA', 'N14.0', 'M26.10', 'Z29.13', 'E81.59', 'G97.1', 'O28.1', 'F45.9', 'S20.01XA', 'Q87.19', 'T82.42XA', 'V23.9XXD', 'L01.0', 'M87.30', 'S60.551A', 'V28.4', 'S02.400B', 'V00.818A', 'S99.922S', 'S92.302G', 'H40.31X0', 'M50.81', 'F98.5', 'V98.8XXS', 'T25.211S', 'S63.610A', 'M00.861', 'T38.2X5A', 'S82.209D', 'S63.219D', 'Z96.29', 'O9A.511', 'J10.83', 'Z90.712', 'M65.251', 'C71.8', 'S40.871A', 'S53.32XD', 'T82.828A', 'N28.86', 'S82.51XB', 'L74.3', 'S32.312A', '

In [30]:
# Count the number of items in the list
number_of_items = len(modified_comorbidity_list)

# Print the result
print(f'The number of items in modified_comorbidity_list is: {number_of_items}')

The number of items in modified_comorbidity_list is: 47374


In [31]:
# Assuming you have a modified_comorbidity_list with values like "123.456" and you want to keep only "123"

# Create a new list with values before the floating point
new_list = [item.split(".")[0] for item in modified_comorbidity_list]
print(new_list)

# # Show the list of modified comorbidityid values with only values before the floating point
# print("Modified Comorbidity ID List with Values Before the Floating Point:")
# for comorbidityid in new_list:
#     print(comorbidityid)


['N17', 'R53', 'R79', 'Z95', 'S27', 'Z34', 'S61', 'Z87', 'O26', 'R15', 'O98', 'S32', 'I67', 'R41', 'V49', 'G93', 'Z21', 'N92', 'M50', 'F90', 'S82', 'N94', 'S98', 'Y93', 'H52', 'S70', 'Q35', 'M67', 'S92', 'V15', 'C60', 'Z97', 'V89', 'S86', 'V45', 'Z03', 'E81', 'G52', 'M23', 'T85', 'Y24', 'T24', 'I82', 'S12', 'Y92', 'T85', 'T46', 'J68', 'P76', 'P55', 'T43', 'S01', 'N14', 'M26', 'Z29', 'E81', 'G97', 'O28', 'F45', 'S20', 'Q87', 'T82', 'V23', 'L01', 'M87', 'S60', 'V28', 'S02', 'V00', 'S99', 'S92', 'H40', 'M50', 'F98', 'V98', 'T25', 'S63', 'M00', 'T38', 'S82', 'S63', 'Z96', 'O9A', 'J10', 'Z90', 'M65', 'C71', 'S40', 'S53', 'T82', 'N28', 'S82', 'L74', 'S32', 'Y73', 'S63', 'M23', 'T84', 'S82', 'D48', 'S12', 'E74', 'G57', 'S90', 'W13', 'L97', 'C81', 'S36', 'S61', 'O04', 'T55', 'T54', 'S00', 'S31', 'H40', 'S13', 'H51', 'Q30', 'C81', 'S42', 'S82', 'S92', 'S82', 'S64', 'G44', 'S67', 'R19', 'E82', 'S63', 'Z00', 'S62', 'E70', 'M89', 'S82', 'S63', 'I97', 'S92', 'E10', 'S52', 'W61', 'S62', 'P78', 'S85'

In [32]:
# Use set to get unique values
unique_values = set(new_list)

# Print unique values
print("Unique Values:")
for value in unique_values:
    print(value)

# Print the total number of unique values
total_unique_count = len(unique_values)
print("\nTotal Number of Unique Values:", total_unique_count)


Unique Values:
C40
Q51
L97
B46
K43
V88
I5A
B52
V21
K44
Z77
Y00
F16
P13
T47
Z85
N86
F45
F93
F03
D09
Z45
A86
Q91
G05
B49
W94
B94
N63
Q67
G92
T25
V35
C52
T27
X52
H05
P96
NKP
F11
V15
B69
W39
K87
H43
W38
G11
H82
A51
F78
Y71
R71
A30
D74
W53
B99
K51
C83
L63
Y25
B66
F02
A33
R91
C75
E66
O02
R34
R20
M42
J43
R85
D36
C66
T34
I60
E78
M33
B95
I28
A20
D13
S00
W20
A52
T31
M66
A39
V91
E24
Y77
d05
X05
O01
I10
S37
Y73
Z22
Q40
O28
F04
P27
I38
I00
T86
R16
T69
V04
S82
S19
F60
G70
K04
H42
H81
M94
r50
W57
F-6
W18
G96
A17
L75
I20
S26
K00
D65
D27
G54
J37
Z63
D45
Z88
Y99
G06
G71
I82
Z68
H74
W46
F68
W19
P70
F21
K06
B35
C11
R49
K13
Z49
I78
R52
K30
E16
N08
S30
P72
j45
B58
L82
Y22
L99
B90
J06
N77
S98
L30
V29
W54
I81
W17
D30
A43
H62
F84
L95
I76
R46
E65
V48
H36
T33
Q69
K64
C93
M86
V79
A98
R12
S96
R62
u07
C63
Z01
C17
R53
A81
G00
Q17
Y72
R36
Z19
E32
L64
G64
Q03
B67
C71
G36
V68
M17
Z38
A99
Q00
A07
K55
E29
B02
Z98
H16
D52
I42
H22
I97
H57
F94
A55
P09
I25
C69
D42
p79
J11
Q18
H02
Q30
Y80
M47
S81
S69
H04
R73
T07
V46
W88
H83
l

In [24]:
# Count the number of values that contain commas
count_with_commas = sum(1 for item in unique_comorbidity_list if "," in item)

# Print the count
print(f"Number of values with commas: {count_with_commas}")

▸,:,


Number of values with commas: 0


In [25]:
# Create a new list with items that don't contain commas
filtered_list = [item for item in unique_comorbidity_list if "," not in item]

# # Print the filtered list
# print("List items without commas:")
print(filtered_list)
# for item in filtered_list:
#     print(item)

▸,:,


['N179', 'R5382', 'R791', 'Z950', 'S27321A', 'Z3492', 'S61219A', 'Z87440', 'O26833', 'R159', 'O98513', 'S32402A', 'I676', 'R41841', 'V491', 'G930', 'Z21', 'N921', 'M5031', 'F900', 'S82131A', 'N942', 'S98929A', 'Y93E2', 'H5213', 'S7000XA', 'Q357', 'M67432', 'S92919D', 'V1551', 'C609', 'Z972', 'V890XXA', 'S86111D', 'V4575', 'Z0379', 'E8182', 'G527', 'M23306', 'T85193A', 'Y248XXA', 'T24309A', 'I82413', 'S12190K', 'Y92001', 'T8541XA', 'T465X1A', 'J689', 'P761', 'P551', 'T43625A', 'S0120XA', 'N140', 'M2610', 'Z2913', 'E8159', 'G971', 'O281', 'F459', 'S2001XA', 'Q8719', 'T8242XA', 'V239XXD', 'L010', 'M8730', 'S60551A', 'V284', 'S02400B', 'V00818A', 'S99922S', 'S92302G', 'H4031X0', 'M5081', 'F985', 'V988XXS', 'T25211S', 'S63610A', 'M00861', 'T382X5A', 'S82209D', 'S63219D', 'Z9629', 'O9A511', 'J1083', 'Z90712', 'M65251', 'C718', 'S40871A', 'S5332XD', 'T82828A', 'N2886', 'S8251XB', 'L743', 'S32312A', 'Y730', 'S63694A', 'M23302', 'T8460XA', 'S82829A', 'D4862', 'S12690D', 'E749', 'G5771', 'S903',

In [62]:
# Read the CSV file into a pandas DataFrame
# df = pd.read_csv("/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/comorbidity_values3.xls")
df = pd.read_csv("/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/CommorbidityAfterSplitCMM.xls")

▸,:,


ParserError: Error tokenizing data. C error: Expected 1 fields in line 3, saw 2


In [ ]:
df = pd.DataFrame(df)

# Extract the 'Name' column from the DataFrame and convert it to a list
name_list = df['Matching Values'].tolist()

# Print the list of 'Name' values
print(name_list)
# Create a new list with the modified comorbidityid values
modified_comorbidity_list1 = [item[:3] + "." + item[3:] for item in name_list]
print(modified_comorbidity_list1)

# Create a new list with values before the floating point
new_list1 = [item.split(".")[0] for item in modified_comorbidity_list1]
print(new_list1)

# Use set to get unique values
unique_values1 = set(new_list1)

# Print unique values
print("Unique Values:")
for value in unique_values1:
    print(value)

# Print the total number of unique values
total_unique_count1 = len(unique_values1)
print("\nTotal Number of Unique Values:", total_unique_count1)

In [ ]:
#Test1 - Matched Records
from pyspark.sql.functions import col, udf, lit, when
from pyspark.sql.types import StringType
from pyspark.sql import functions as F

# Define a Python function to convert comorbidityid
def convert_comorbidityid(comorbidityid):
    # Check if comorbidityid contains a decimal point
    if '.' in comorbidityid:
        parts = comorbidityid.split('.')
        key = parts[0]
        mapped_value = icd_dict.get(key, None)
        if mapped_value:
            mapped_value += '.' + '.'.join(parts[1:])
            return mapped_value
    else:
        # Preserve the original value if not found in the dictionary
        mapped_value = icd_dict.get(comorbidityid, None)
        if mapped_value is not None:
            return mapped_value

    # If no mapped value is found or no decimal point, preserve the original value
    return comorbidityid

# Register the convert_comorbidityid function as a UDF
convert_comorbidityid_udf = udf(convert_comorbidityid, StringType())

# Apply the UDF to replace comorbidityid values
result_df = Como_Result_Final1.withColumn("comorbidityid", convert_comorbidityid_udf(col("comorbidityid")))

# Calculate the count of records where comorbidityid matches before the floating point
result_df = result_df.withColumn("matched_count", when(col("comorbidityid") != col("comorbidityid"), lit(1)).otherwise(lit(0)))

# Calculate the total count of matched records before the floating point
total_matched_count = result_df.agg(F.sum("matched_count")).collect()[0][0]

# Show the resulting DataFrame
result_df.show(truncate=False)

# Show the total count of matched records before the floating point
print("Total Matched Count Before Floating Point:", total_matched_count)


In [ ]:
# # Extract dictionary values
# dictionary_values = icd_dict.values()

# # Create lists to store matching and unmatched values
# matching_values = []
# unmatched_values = []

# # Iterate through comorbidity IDs and classify them
# for value in unique_comorbidity_list:
#     if value in dictionary_values:
#         matching_values.append(value)
#     else:
#         unmatched_values.append(value)

# # Calculate the counts of matched and unmatched values
# matching_count = len(matching_values)
# unmatched_count = len(unmatched_values)

# # Print the matching and unmatched values along with their counts
# print("Matching Values:", matching_values)
# print("Unmatched Values:", unmatched_values)
# print("Count of Matching Values:", matching_count)
# print("Count of Unmatched Values:", unmatched_count)

import csv

# Extract dictionary values
dictionary_values = icd_dict.values()

# Create lists to store matching and unmatched values
matching_values = []
unmatched_values = []

# Iterate through comorbidity IDs and classify them
for value in filtered_list:
    if value in dictionary_values:
        matching_values.append(value)
    else:
        unmatched_values.append(value)

# Calculate the counts of matched and unmatched values
matching_count = len(matching_values)
unmatched_count = len(unmatched_values)

# Specify the path where you want to save the CSV file
csv_file_path = "/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/comorbidity_values3.xls"

# Create a CSV file and write the values
with open(csv_file_path, mode="w", newline="") as csvfile:
    fieldnames = ["Matching Values", "Unmatched Values"]
    writer = csv.writer(csvfile)

    # Write the headers
    writer.writerow(fieldnames)

    # Write the matching and unmatched values
    for match, unmatch in zip(matching_values, unmatched_values):
        writer.writerow([match, unmatch])

print("Count of Matching Values:", matching_count)
print("Count of Unmatched Values:", unmatched_count)

In [ ]:
# # Extract dictionary values and compare with manually created combined list

import csv

# Extract dictionary values
dictionary_values = icd_dict.values()

# Create lists to store matching and unmatched values
matching_values = []
unmatched_values = []

# Iterate through comorbidity IDs and classify them
for value in combined_and_distinct:
    if value in dictionary_values:
        matching_values.append(value)
    else:
        unmatched_values.append(value)

# Calculate the counts of matched and unmatched values
matching_count = len(matching_values)
unmatched_count = len(unmatched_values)

# Specify the path where you want to save the CSV file
csv_file_path = "/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Manual_comorbidity_values.xls"

# Create a CSV file and write the values
with open(csv_file_path, mode="w", newline="") as csvfile:
    fieldnames = ["Matching Values", "Unmatched Values"]
    writer = csv.writer(csvfile)

    # Write the headers
    writer.writerow(fieldnames)

    # Write the matching and unmatched values
    for match, unmatch in zip(matching_values, unmatched_values):
        writer.writerow([match, unmatch])

print("Count of Matching Values:", matching_count)
print("Count of Unmatched Values:", unmatched_count)